# RFC Simulator and Paper-Reproduction Notebook

**Binder-safe patch of the original simulator.** This version keeps the dropdown UI when `ipywidgets` is available, removes the `pandas` dependency, and searches both the repository root and `notebooks/` for the JSON files.

It supports two clearly separated paths:

1. **Current paper-reproduction path**: G, R, N, DownstreamPhysicalProjection, DimensionlessValidation, S, T, U, V, W, X, Y2, Z, QG.
2. **Legacy/development simulator path**: previous simulator modules A through Q, plus legacy G/N/R modules when present.

Expected files:

- `SimulationConfigs.json`
- `Module_G_R_N_S_T_FrozenPacket.json`
- `ValidationScreens_U_V_W_X_Y2_Z_QG.json` optional but recommended

Global rule for the paper-reproduction path: modules consume the frozen packet. They do not retune it.


In [ ]:
import json
import math
from pathlib import Path

from IPython.display import display, Markdown, HTML

try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except Exception:
    widgets = None
    WIDGETS_AVAILABLE = False

BASE_DIR = Path.cwd()

EXPECTED_FILES = {
    "simulation_configs": "SimulationConfigs.json",
    "frozen_packet": "Module_G_R_N_S_T_FrozenPacket.json",
    "validation_screens": "ValidationScreens_U_V_W_X_Y2_Z_QG.json"
}

PAPER_REPRODUCTION_ORDER = [
    "G", "R", "N", "DownstreamPhysicalProjection", "DimensionlessValidation",
    "S", "T", "U", "V", "W", "X", "Y2", "Z", "QG"
]

LEGACY_DEVELOPMENT_ORDER = [
    "A", "B", "C", "D", "E", "F", "H", "I", "J", "K", "L", "M", "O", "P", "Q",
    "G_legacy", "N_legacy", "R_legacy"
]

LEGACY_MODULES = {"G_legacy", "N_legacy", "R_legacy"}
CANONICAL_ORDER = PAPER_REPRODUCTION_ORDER

display(Markdown("## 1. Load repository JSON files"))
print("Working directory:", BASE_DIR)
print("Binder-safe patch: pandas removed; dropdown preserved when ipywidgets is available.")


In [ ]:
def html_escape(value):
    text = str(value)
    return (text.replace("&", "&amp;")
                .replace("<", "&lt;")
                .replace(">", "&gt;")
                .replace('"', "&quot;"))


def shorten(value, max_len=900):
    if isinstance(value, (dict, list)):
        value = json.dumps(value, ensure_ascii=False, indent=2)
    else:
        value = str(value)
    if len(value) > max_len:
        return value[:max_len] + " ..."
    return value


def display_title(title):
    display(Markdown(f"## {title}"))


def display_note(text):
    display(Markdown(text))


def display_table(rows, max_rows=200):
    """Display rows as an HTML table without pandas."""
    if rows is None:
        display_note("`None`")
        return
    if isinstance(rows, dict):
        rows = [{"field": k, "value": v} for k, v in rows.items()]
    if not isinstance(rows, list):
        display_note(f"`{html_escape(shorten(rows))}`")
        return
    if not rows:
        display_note("_No rows to display._")
        return

    normalized = []
    for item in rows[:max_rows]:
        normalized.append(item if isinstance(item, dict) else {"value": item})

    columns = []
    for row in normalized:
        for key in row.keys():
            if key not in columns:
                columns.append(key)

    html = ["<div style='overflow-x:auto; max-width:100%;'>"]
    html.append("<table style='border-collapse:collapse; width:100%; font-size:14px;'>")
    html.append("<thead><tr>")
    for col in columns:
        html.append(f"<th style='border:1px solid #bbb; padding:6px; background:#eee; text-align:left;'>{html_escape(col)}</th>")
    html.append("</tr></thead><tbody>")
    for row in normalized:
        html.append("<tr>")
        for col in columns:
            value = html_escape(shorten(row.get(col, "")))
            html.append(f"<td style='border:1px solid #bbb; padding:6px; vertical-align:top; white-space:pre-wrap;'>{value}</td>")
        html.append("</tr>")
    html.append("</tbody></table></div>")
    if len(rows) > max_rows:
        html.append(f"<p>Showing first {max_rows} rows of {len(rows)} total rows.</p>")
    display(HTML("".join(html)))


def display_dataframe_safe(rows, max_rows=200):
    # Compatibility name retained from the old notebook. No pandas required.
    display_table(rows, max_rows=max_rows)


In [ ]:
def candidate_file_paths(filename):
    cwd = Path.cwd()
    candidates = [
        cwd / filename,
        cwd / "notebooks" / filename,
        cwd.parent / filename,
        cwd.parent / "notebooks" / filename,
    ]
    try:
        candidates.extend(list(cwd.rglob(filename)))
    except Exception:
        pass
    seen = set()
    unique = []
    for path in candidates:
        key = str(path.resolve()) if path.exists() else str(path)
        if key not in seen:
            seen.add(key)
            unique.append(path)
    return unique


def locate_file(filename):
    for path in candidate_file_paths(filename):
        if path.exists() and path.is_file():
            return path
    return None


def load_json_file(filename, required=False):
    path = locate_file(filename)
    if path is None:
        if required:
            raise FileNotFoundError(f"Required file not found: {filename}")
        return None, {"file": filename, "path": "not found", "exists": False, "loaded": False, "error": None}
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        return data, {"file": filename, "path": str(path), "exists": True, "loaded": True, "error": None}
    except Exception as exc:
        if required:
            raise
        return None, {"file": filename, "path": str(path), "exists": True, "loaded": False, "error": str(exc)}


simulation_configs, simulation_status = load_json_file(EXPECTED_FILES["simulation_configs"])
frozen_packet, frozen_status = load_json_file(EXPECTED_FILES["frozen_packet"])
validation_file, validation_status = load_json_file(EXPECTED_FILES["validation_screens"])

display_table([simulation_status, frozen_status, validation_status])

if frozen_packet is None and simulation_configs is None:
    raise RuntimeError("No usable RFC JSON files were found. Put the JSON files in the repo root or notebooks/ folder.")

display(Markdown("Loaded files. Continuing with schema normalization."))


In [ ]:
def as_list(x):
    if x is None:
        return []
    return x if isinstance(x, list) else [x]


def normalize_name(name):
    return str(name).lower().replace("_", "").replace("-", "").replace(" ", "")


def unwrap_modules(config):
    if config is None:
        return []
    if isinstance(config, list):
        return config
    if isinstance(config, dict):
        for key in ["modules", "moduleConfigs", "simulationModules", "configs"]:
            if isinstance(config.get(key), list):
                return config[key]
    return []


simulation_modules = unwrap_modules(simulation_configs)


def module_name(module_obj):
    if not isinstance(module_obj, dict):
        return None
    for key in ["module", "id", "name", "moduleId", "moduleID"]:
        value = module_obj.get(key)
        if isinstance(value, str):
            return value
    return None


def available_config_module_names():
    return [name for obj in simulation_modules for name in [module_name(obj)] if name is not None]


def find_module_in_configs(name):
    target = normalize_name(name)
    for obj in simulation_modules:
        obj_name = module_name(obj)
        if obj_name and normalize_name(obj_name) == target:
            return obj
    return None


def find_key_contains(obj, tokens):
    if obj is None or not isinstance(obj, dict):
        return None
    tokens_low = [str(t).lower() for t in tokens]
    for key, value in obj.items():
        key_low = str(key).lower()
        if all(t in key_low for t in tokens_low):
            return value
    return None


def get_packet_module(name):
    if frozen_packet is None or not isinstance(frozen_packet, dict):
        return None
    target = normalize_name(name)
    direct_candidates = [name, str(name).lower(), f"module{name}", f"module_{name}", f"module{str(name).lower()}", f"module_{str(name).lower()}"]
    for key in direct_candidates:
        if key in frozen_packet:
            return frozen_packet[key]
    for key, value in frozen_packet.items():
        key_clean = normalize_name(key)
        if key_clean == target or key_clean == normalize_name("module" + str(name)):
            return value
        if key_clean.startswith(normalize_name("module" + str(name))):
            return value
    if target == "n":
        return find_key_contains(frozen_packet, ["module", "n"]) or find_key_contains(frozen_packet, ["dimensional"])
    if target == "r":
        return find_key_contains(frozen_packet, ["module", "r"]) or find_key_contains(frozen_packet, ["closure", "audit"])
    if target == "s":
        return find_key_contains(frozen_packet, ["module", "s"]) or find_key_contains(frozen_packet, ["anchor"])
    if target == "t":
        return find_key_contains(frozen_packet, ["module", "t"]) or find_key_contains(frozen_packet, ["coupling"])
    return None


def raw_validation_block():
    if isinstance(validation_file, dict):
        if isinstance(validation_file.get("validationScreens"), dict):
            return validation_file.get("validationScreens")
        return validation_file
    if isinstance(frozen_packet, dict) and isinstance(frozen_packet.get("validationScreens"), dict):
        return frozen_packet.get("validationScreens")
    return {}


def infer_screen_id(key):
    clean = normalize_name(key)
    if clean in ["u", "v", "w", "x", "y2", "z", "qg"]:
        return clean.upper()
    for candidate in ["Y2", "QG", "U", "V", "W", "X", "Z"]:
        c = candidate.lower()
        if clean.startswith(c) or clean.startswith("module" + c) or ("module" + c) in clean:
            return candidate
    return str(key)


def normalize_validation_screens():
    raw = raw_validation_block()
    normalized = {}
    if isinstance(raw, dict):
        for key, value in raw.items():
            normalized[infer_screen_id(key)] = value
    for name in ["U", "V", "W", "X", "Y2", "Z", "QG"]:
        if name not in normalized:
            cfg = find_module_in_configs(name)
            if cfg is not None:
                normalized[name] = cfg
    return normalized


validation_screens = normalize_validation_screens()


def get_downstream_projection():
    if isinstance(frozen_packet, dict):
        for key in ["downstreamPhysicalProjectionScreen", "downstreamPhysicalProjection", "combinedDownstreamPhysicalProjectionScreen"]:
            if key in frozen_packet:
                return frozen_packet[key]
        found = find_key_contains(frozen_packet, ["downstream", "projection"])
        if found is not None:
            return found
    return find_module_in_configs("DownstreamPhysicalProjection")


def get_dimensionless_validation():
    if isinstance(frozen_packet, dict):
        for key in ["dimensionlessValidationLayer", "dimensionlessValidation", "dimensionlessObservableValidationLayer"]:
            if key in frozen_packet:
                return frozen_packet[key]
        found = find_key_contains(frozen_packet, ["dimensionless", "validation"])
        if found is not None:
            return found
    return find_module_in_configs("DimensionlessValidation")


def get_module_data(name):
    if name == "DownstreamPhysicalProjection":
        return get_downstream_projection()
    if name == "DimensionlessValidation":
        return get_dimensionless_validation()
    if name in ["U", "V", "W", "X", "Y2", "Z", "QG"]:
        return validation_screens.get(name) or find_module_in_configs(name)
    if name in ["G", "R", "N", "S", "T"]:
        return get_packet_module(name) or find_module_in_configs(name)
    return find_module_in_configs(name) or get_packet_module(name)


def build_all_runnable_modules():
    config_names = available_config_module_names()
    ordered = []
    for name in PAPER_REPRODUCTION_ORDER:
        if name not in ordered:
            ordered.append(name)
    for name in LEGACY_DEVELOPMENT_ORDER:
        if any(normalize_name(name) == normalize_name(cn) for cn in config_names) and name not in ordered:
            ordered.append(name)
    for name in config_names:
        if name not in ordered:
            ordered.append(name)
    return ordered


ALL_RUNNABLE_MODULES = build_all_runnable_modules()

display(Markdown("## 2. Normalized package status"))
summary_rows = []
for name in PAPER_REPRODUCTION_ORDER:
    summary_rows.append({"path": "paper_reproduction", "component": name, "found": get_module_data(name) is not None, "source": "frozen/validation/config auto-detected"})
for name in LEGACY_DEVELOPMENT_ORDER:
    if get_module_data(name) is not None:
        summary_rows.append({"path": "legacy_development", "component": name, "found": True, "source": "SimulationConfigs.json"})
display_table(summary_rows)
display(Markdown("Runnable module selector includes paper modules, legacy modules, and any extra modules found in `SimulationConfigs.json`."))


In [ ]:
def flatten_dict(obj, prefix=""):
    rows = []
    if isinstance(obj, dict):
        for key, value in obj.items():
            next_prefix = f"{prefix}.{key}" if prefix else str(key)
            if isinstance(value, dict):
                rows.extend(flatten_dict(value, next_prefix))
            elif isinstance(value, list):
                if all(isinstance(x, dict) for x in value):
                    rows.append({"field": next_prefix, "value": f"list[{len(value)}] of records"})
                else:
                    rows.append({"field": next_prefix, "value": value})
            else:
                rows.append({"field": next_prefix, "value": value})
    else:
        rows.append({"field": prefix or "value", "value": obj})
    return rows


def find_first_record_list(obj):
    if isinstance(obj, list) and all(isinstance(x, dict) for x in obj):
        return obj
    if isinstance(obj, dict):
        for key in ["results", "screenResults", "selectedResults", "scoredConstants", "constants", "rows", "quarks", "mixingAngles", "observerBranchingResults", "neuralEEGTargets"]:
            if key in obj:
                found = find_first_record_list(obj[key])
                if found is not None:
                    return found
        for value in obj.values():
            found = find_first_record_list(value)
            if found is not None:
                return found
    return None


def extract_result_container(obj):
    if not isinstance(obj, dict):
        return obj
    for key in ["results", "screenResults", "selectedResults", "scoredConstants", "constants", "summary"]:
        if key in obj:
            return obj[key]
    return obj


def display_object(obj, title="Object"):
    display_title(title)
    if obj is None:
        display_note("**Not found.** Check that the relevant JSON file is present and populated.")
        return
    record_list = find_first_record_list(obj)
    if record_list is not None and len(record_list) > 0:
        display_table(record_list)
    container = extract_result_container(obj)
    if isinstance(container, dict):
        display_table(flatten_dict(container))
    elif isinstance(container, list):
        display_table(container)
    else:
        display(container)


def numeric_or_none(value):
    try:
        if isinstance(value, bool):
            return None
        return float(value)
    except Exception:
        return None


def deep_get_by_key(obj, target_keys):
    target_low = {str(k).lower() for k in as_list(target_keys)}
    if isinstance(obj, dict):
        for key, value in obj.items():
            if str(key).lower() in target_low:
                return value
        for value in obj.values():
            found = deep_get_by_key(value, target_keys)
            if found is not None:
                return found
    elif isinstance(obj, list):
        for value in obj:
            found = deep_get_by_key(value, target_keys)
            if found is not None:
                return found
    return None


def get_claim_boundary(obj):
    if isinstance(obj, dict):
        for key in ["claimBoundary", "boundary", "claimBoundarySummary", "interpretation", "status"]:
            if key in obj:
                return obj[key]
    return None


def print_boundary_if_present(obj):
    boundary = get_claim_boundary(obj)
    if boundary is not None:
        display_note("**Claim boundary / interpretation / status:**")
        display(boundary)


In [ ]:
def module_g_check():
    data = get_module_data("G")
    display_title("Module G: Deterministic Triadic Closure")
    if data is None:
        display_note("**Module G not found.**")
        return
    delta = numeric_or_none(deep_get_by_key(data, ["delta"]))
    cycle_length = numeric_or_none(deep_get_by_key(data, ["cycleLength", "cycle_length"]))
    phase_depth_k = numeric_or_none(deep_get_by_key(data, ["phaseDepthK", "phase_depth_k"]))
    alpha_packet = numeric_or_none(deep_get_by_key(data, ["alpha"]))
    nu_packet = numeric_or_none(deep_get_by_key(data, ["nu"]))
    epsilon_packet = numeric_or_none(deep_get_by_key(data, ["epsilon"]))
    empirical_targets = deep_get_by_key(data, ["empiricalTargetsUsed", "empirical_targets_used"])
    rows = []
    if delta is not None and cycle_length is not None and alpha_packet is not None:
        alpha_expected = math.log(delta) / cycle_length
        rows.append({"check": "alpha = log(delta) / cycleLength", "expected": alpha_expected, "packet": alpha_packet, "absError": abs(alpha_expected - alpha_packet)})
    if delta is not None and phase_depth_k is not None and nu_packet is not None:
        nu_expected = phase_depth_k * delta ** (-4)
        rows.append({"check": "nu = phaseDepthK * delta^(-4)", "expected": nu_expected, "packet": nu_packet, "absError": abs(nu_expected - nu_packet)})
    if alpha_packet is not None and nu_packet is not None and epsilon_packet is not None:
        epsilon_expected = alpha_packet * nu_packet
        rows.append({"check": "epsilon = alpha * nu", "expected": epsilon_expected, "packet": epsilon_packet, "absError": abs(epsilon_expected - epsilon_packet)})
    rows.append({"check": "empiricalTargetsUsed", "expected": False, "packet": empirical_targets, "absError": 0 if empirical_targets is False else "CHECK"})
    display_table(rows)
    display_object(data, "Module G Stored Packet")


def module_r_check():
    data = get_module_data("R")
    display_title("Module R: Triad-Grouped Global Closure Audit")
    if data is None:
        display_note("**Module R not found.** This is a packaging problem if the paper reports Module R values.")
        return
    diagnostic_keys = ["rawRFLResidualScore", "sourceCoupledRFLResidualScore", "residualImprovement", "rawStandardizedResidualScore", "sourceCoupledRFLResidualScoreV2", "residualImprovementV2", "moduleRScoreV2", "cpResidual", "tailN18", "tailN40", "bestLagCorrelation", "bestLagCorrelationV1"]
    rows = []
    for key in diagnostic_keys:
        value = deep_get_by_key(data, [key])
        if value is not None:
            rows.append({"field": key, "value": value})
    if rows:
        display_table(rows)
    placeholder_flags = []
    for bad_key, bad_value in [("rawRFLResidualScore", 0.5), ("sourceCoupledRFLResidualScore", 0.5), ("residualImprovement", 0.0), ("tailN18", 0.0), ("tailN40", 0.0)]:
        val = numeric_or_none(deep_get_by_key(data, [bad_key]))
        if val is not None and abs(val - bad_value) < 1e-15:
            placeholder_flags.append(bad_key)
    if placeholder_flags:
        display_note("**WARNING:** possible placeholder Module R values detected: " + ", ".join(placeholder_flags))
    else:
        display_note("No obvious placeholder Module R values detected.")
    display_object(data, "Full Module R Object")


def generic_module_check(name, title=None):
    data = get_module_data(name)
    display_object(data, title or f"Module {name}")
    if name in LEGACY_MODULES or name in LEGACY_DEVELOPMENT_ORDER:
        display_note("**Legacy/development note:** This module is retained for continuity and exploration. It is not part of the current G/R/N/S/T paper-reproduction spine unless explicitly stated in the paper.")
    print_boundary_if_present(data)


def validation_screen_check(name):
    data = get_module_data(name)
    titles = {"U": "Module U: One-Anchor Constant Table Screen", "V": "Module V: Precision Cosmology Compressed-Parameter Screen", "W": "Module W: BBN Light-Abundance Proxy Screen", "X": "Module X: CP/EDM Bound Screen", "Y2": "Module Y2: Particle-Sector Refinement Screen", "Z": "Module Z: Observer, Branching, Neural, and EEG Harness", "QG": "Module QG: Finite Spin-Foam Transition-Amplitude Audit"}
    display_object(data, titles.get(name, f"Module {name}"))
    if name == "Y2":
        display_note("**Important:** Y2 is exploratory candidate discovery. It is not final independent validation until frozen and retested as Y3.")
    print_boundary_if_present(data)


def run_component(name):
    if name == "G":
        module_g_check()
    elif name == "R":
        module_r_check()
    elif name == "N":
        generic_module_check("N", "Module N V2: Dimensional Projection Bridge")
    elif name == "S":
        generic_module_check("S", "Module S: One-Anchor SI Bridge")
    elif name == "T":
        generic_module_check("T", "Module T: Dimensionless Coupling Map")
    elif name == "DownstreamPhysicalProjection":
        generic_module_check("DownstreamPhysicalProjection", "Downstream Physical-Projection Screen")
    elif name == "DimensionlessValidation":
        generic_module_check("DimensionlessValidation", "Dimensionless Observable Validation Layer")
    elif name in ["U", "V", "W", "X", "Y2", "Z", "QG"]:
        validation_screen_check(name)
    else:
        generic_module_check(name, f"Module {name}")


def run_all_paper():
    display(Markdown("# RFC Full Paper-Reproduction Pass"))
    for name in PAPER_REPRODUCTION_ORDER:
        run_component(name)
        display(HTML("<hr>"))


def run_legacy_development_modules():
    display(Markdown("# RFC Legacy / Development Simulator Pass"))
    ran_any = False
    for name in LEGACY_DEVELOPMENT_ORDER:
        data = get_module_data(name)
        if data is not None:
            ran_any = True
            run_component(name)
            display(HTML("<hr>"))
    if not ran_any:
        display_note("No legacy/development modules were found in SimulationConfigs.json.")


def run_all():
    run_all_paper()


display(Markdown("## 3. Module runners loaded"))
display(Markdown("Use `run_all_paper()` for the current paper path, `run_legacy_development_modules()` for A-Q and legacy G/N/R, or the dropdown below."))


In [ ]:
def package_audit():
    display_title("Repository Package Audit")
    rows = []
    rows.append({"check": "SimulationConfigs.json loaded", "status": simulation_configs is not None})
    rows.append({"check": "Frozen packet JSON loaded", "status": frozen_packet is not None})
    rows.append({"check": "Standalone validation screens JSON loaded", "status": validation_file is not None})
    for name in ["G", "R", "N", "S", "T", "DownstreamPhysicalProjection", "DimensionlessValidation", "U", "V", "W", "X", "Y2", "Z", "QG"]:
        rows.append({"check": f"{name} found", "status": get_module_data(name) is not None})
    legacy_found = [name for name in LEGACY_DEVELOPMENT_ORDER if get_module_data(name) is not None]
    rows.append({"check": "Legacy/development modules found", "status": len(legacy_found) > 0})
    display_table(rows)
    missing = [row["check"] for row in rows if row.get("status") is False]
    if missing:
        display_note("**Packaging warnings:**")
        for item in missing:
            display_note("- " + item)
    else:
        display_note("**Package audit passed:** all expected paper-reproduction components were found.")
    if validation_file is None:
        display_note("**Recommendation:** add `ValidationScreens_U_V_W_X_Y2_Z_QG.json` as a standalone file, even if the same screens are mirrored in the frozen packet.")
    if legacy_found:
        display_note("**Legacy/development modules detected:** " + ", ".join(legacy_found))
    else:
        display_note("No legacy/development modules detected. This is okay only if SimulationConfigs.json intentionally omits them.")


package_audit()


In [ ]:
if WIDGETS_AVAILABLE:
    display(Markdown("## 4. Interactive module selector"))
    selector = widgets.Dropdown(
        options=ALL_RUNNABLE_MODULES,
        value="G" if "G" in ALL_RUNNABLE_MODULES else ALL_RUNNABLE_MODULES[0],
        description="Component:",
        layout=widgets.Layout(width="620px")
    )
    run_button = widgets.Button(description="Run selected component", button_style="primary")
    run_paper_button = widgets.Button(description="Run paper reproduction", button_style="success")
    run_legacy_button = widgets.Button(description="Run legacy/development", button_style="warning")
    output = widgets.Output()

    def on_run_clicked(_):
        with output:
            output.clear_output()
            run_component(selector.value)

    def on_run_paper_clicked(_):
        with output:
            output.clear_output()
            run_all_paper()

    def on_run_legacy_clicked(_):
        with output:
            output.clear_output()
            run_legacy_development_modules()

    run_button.on_click(on_run_clicked)
    run_paper_button.on_click(on_run_paper_clicked)
    run_legacy_button.on_click(on_run_legacy_clicked)

    display(widgets.VBox([selector, widgets.HBox([run_button, run_paper_button, run_legacy_button])]))
    display(output)
else:
    display(Markdown("## 4. Interactive widgets unavailable"))
    display(Markdown("The dropdown requires `ipywidgets`. Manual commands still work: `run_component('G')`, `run_all_paper()`, or `run_legacy_development_modules()`."))


## Manual commands

```python
package_audit()
run_all_paper()
run_component('G')
run_component('R')
run_component('N')
run_component('DownstreamPhysicalProjection')
run_component('DimensionlessValidation')
run_component('S')
run_component('T')
run_component('U')
run_component('V')
run_component('W')
run_component('X')
run_component('Y2')
run_component('Z')
run_component('QG')
run_legacy_development_modules()
run_component('A')
run_component('B')
run_component('Q')
run_component('G_legacy')
run_component('N_legacy')
run_component('R_legacy')
```

Interpretation reminders:

- Module G is the current frozen deterministic packet source.
- Module R audits the frozen packet; it does not create the packet.
- Module U is a first-pass electromagnetic/atomic constant table screen.
- Module Y2 is exploratory candidate discovery and must be frozen/retested as Y3 before independent validation claims.
- A-Q and legacy G/N/R modules are retained for development continuity and should not be confused with the current paper-reproduction spine.
